# 🔍 Column Comparison Tool for Catalog Datasets
This notebook compares whether two dataset files (CSV, FITS, Parquet, TSV) in the `data/` directory have the exact same columns.

If they do not have the same columns, it clearly displays:
- **Which columns are exclusive to File 1** (missing from File 2)
- **Which columns are exclusive to File 2** (missing from File 1)
- **Which columns are shared in common**
- **Whether the ordering of common columns matches or differs**
- **A side-by-side alignment table** showing the status and position of every column.

---
### 🚀 Quick Start:
1. Run **Cell 1** to initialize imports and setup.
2. In **Cell 5**, set your two files in the `FILE_1` and `FILE_2` variables (or pick from the catalog list in Cell 4).
3. Run **Cell 5** to see the comparison verdict, summary statistics, and column diff tables!


In [1]:
import os
import re
import csv
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any

# Data handling and display
try:
    import pandas as pd
    HAS_PANDAS = True
    pd.set_option('display.max_rows', 100)
    pd.set_option('display.max_columns', 20)
    pd.set_option('display.width', 1000)
except ImportError:
    HAS_PANDAS = False
    pd = None

from IPython.display import display, HTML, Markdown

# Astropy for FITS files (used if installed)
try:
    from astropy.io import fits
    HAS_ASTROPY = True
except ImportError:
    HAS_ASTROPY = False

print("✅ Setup complete. Ready to compare datasets.")


✅ Setup complete. Ready to compare datasets.


## 🛠️ 1. File Discovery & Efficient Column Extraction
The functions below:
- Resolve file paths seamlessly whether the notebook is executed from `code/` or the repository root.
- Extract column names **efficiently in milliseconds without loading multi-gigabyte data into memory** (using `nrows=0` for CSV and binary header parsing for FITS).


In [2]:
def resolve_file_path(filepath: str | Path) -> Path:
    """
    Resolves a file path whether running from 'code/' or from repository root.
    """
    p = Path(filepath)
    if p.is_file():
        return p.resolve()
    
    cwd = Path.cwd()
    # Check relative to cwd
    if (cwd / p).is_file():
        return (cwd / p).resolve()
    
    # Check relative to parent directory (repo root when cwd is code/)
    if (cwd.parent / p).is_file():
        return (cwd.parent / p).resolve()
        
    # Check if ../path can be resolved from repo root
    if p.parts and p.parts[0] == '..':
        rel = Path(*p.parts[1:])
        if (cwd / rel).is_file():
            return (cwd / rel).resolve()
        if (cwd.parent / rel).is_file():
            return (cwd.parent / rel).resolve()
            
    # Check under data/ directory
    for candidate_dir in [cwd / 'data', cwd.parent / 'data', cwd / '../data']:
        if (candidate_dir / p).is_file():
            return (candidate_dir / p).resolve()
        if (candidate_dir / p.name).is_file():
            return (candidate_dir / p.name).resolve()
            
    return p


def get_data_dir() -> Path:
    """Finds and returns the absolute path to the data/ directory."""
    cwd = Path.cwd()
    for candidate in [cwd / 'data', cwd / '../data', cwd.parent / 'data', cwd / '../../data']:
        if candidate.is_dir():
            return candidate.resolve()
    return cwd / 'data'


def list_data_files(data_dir: Optional[str | Path] = None, show_table: bool = True) -> list[Path]:
    """
    Scans the data/ folder and lists all available tabular catalog files (.csv, .fits, etc.).
    """
    folder = Path(data_dir) if data_dir else get_data_dir()
    if not folder.is_dir():
        print(f"⚠️ Data directory not found at: {folder}")
        return []
    
    valid_exts = {'.csv', '.fits', '.fit', '.tsv', '.parquet'}
    files = sorted([
        f for f in folder.glob('**/*') 
        if f.is_file() and f.suffix.lower() in valid_exts and not f.name.endswith('.bak')
    ])
    
    if show_table:
        cwd = Path.cwd()
        records = []
        for i, f in enumerate(files, 1):
            try:
                rel_path = f.relative_to(folder.parent)
            except ValueError:
                rel_path = f
            size_mb = f.stat().st_size / (1024 * 1024)
            path_str = f"../{rel_path}" if cwd.name == 'code' else str(rel_path)
            records.append({
                "#": i,
                "Relative Path": path_str,
                "Filename": f.name,
                "Format": f.suffix.upper()[1:],
                "Size (MB)": f"{size_mb:.2f} MB"
            })
        if HAS_PANDAS:
            df_files = pd.DataFrame(records)
            display(HTML("<h3>📂 Available Tabular Files in <code>data/</code>:</h3>"))
            display(df_files)
        else:
            print("Available Tabular Files:")
            for r in records:
                print(f" {r['#']:>2}. {r['Relative Path']} ({r['Size (MB)']})")
        
    return files


def extract_columns(filepath: str | Path) -> list[str]:
    """
    Extracts the column names from a file (CSV, FITS, Parquet, TSV)
    efficiently without loading the entire dataset into memory.
    """
    path = resolve_file_path(filepath)
    if not path.is_file():
        raise FileNotFoundError(f"File not found: '{filepath}' (resolved to: '{path}')")
        
    ext = path.suffix.lower()
    
    # 1. FITS files (e.g. GALAH catalogues)
    if ext in ['.fits', '.fit']:
        # Try astropy first if installed
        if HAS_ASTROPY:
            try:
                with fits.open(path, memmap=True) as hdul:
                    for hdu in hdul:
                        if hasattr(hdu, 'columns') and hdu.columns is not None:
                            return [str(c).strip() for c in hdu.columns.names]
            except Exception:
                pass
        # Pure Python fallback: read FITS HDU headers looking for TTYPEn keywords
        cols = []
        with open(path, 'rb') as f:
            block_size = 2880
            while True:
                block = f.read(block_size)
                if not block or len(block) < block_size:
                    break
                text = block.decode('latin1', errors='ignore')
                for line in [text[i:i+80] for i in range(0, len(text), 80)]:
                    if line.startswith('TTYPE'):
                        m = re.match(r"TTYPE\d+\s*=\s*'([^']*)'", line)
                        if m:
                            cols.append(m.group(1).strip())
                    elif line.startswith('END '):
                        if cols:
                            return cols
        if cols:
            return cols
        raise ValueError(f"Could not extract table columns from FITS file: {path}")

    # 2. Parquet files
    elif ext == '.parquet':
        try:
            import pyarrow.parquet as pq
            schema = pq.read_schema(path)
            return list(schema.names)
        except Exception:
            return list(pd.read_parquet(path).columns)

    # 3. CSV / TSV / Delimited text files
    else:
        # Fast path with pandas: reading 0 rows only fetches column headers
        if HAS_PANDAS:
            try:
                cols = list(pd.read_csv(path, nrows=0).columns)
                return [str(c).strip() for c in cols]
            except Exception:
                pass
        # Fallback to standard csv module
        sep = '	' if ext == '.tsv' else ','
        with open(path, mode='r', encoding='utf-8', errors='replace') as f:
            reader = csv.reader(f, delimiter=sep)
            header = next(reader)
            return [str(c).strip() for c in header]


## ⚖️ 2. Comparison Engine
The `compare_columns()` function performs a complete analysis:
- **Set Difference**: Finds columns unique to File 1, unique to File 2, and in both.
- **Order Verification**: Checks if common columns share the same column index.
- **Side-by-side Table**: Generates a unified DataFrame with presence and position info.


In [3]:
def compare_columns(
    file1: str | Path, 
    file2: str | Path, 
    display_results: bool = True
) -> dict:
    """
    Compares the columns of two tabular files.
    
    Parameters:
    -----------
    file1 : str or Path
        Path to the first file.
    file2 : str or Path
        Path to the second file.
    display_results : bool, default True
        Whether to display rich HTML / Markdown summary and comparison tables.
        
    Returns:
    --------
    dict containing:
        - 'is_exact_match': bool (same columns in identical order)
        - 'is_same_set': bool (same columns, ignoring order)
        - 'cols1': list of columns in file1
        - 'cols2': list of columns in file2
        - 'common_cols': list of columns in both
        - 'only_in_file1': list of columns only in file1
        - 'only_in_file2': list of columns only in file2
        - 'order_mismatches': list of common columns that appear at different indices
        - 'comparison_df': pandas DataFrame aligning all columns side-by-side
        - 'diff_df': pandas DataFrame containing only the differing columns
    """
    path1 = resolve_file_path(file1)
    path2 = resolve_file_path(file2)
    
    cols1 = extract_columns(path1)
    cols2 = extract_columns(path2)
    
    set1 = set(cols1)
    set2 = set(cols2)
    
    common_cols = [c for c in cols1 if c in set2]
    only_in_file1 = [c for c in cols1 if c not in set2]
    only_in_file2 = [c for c in cols2 if c not in set1]
    
    is_exact_match = (cols1 == cols2)
    is_same_set = (set1 == set2)
    
    # Check 1-based index alignment for common columns
    col1_idx = {c: i + 1 for i, c in enumerate(cols1)}
    col2_idx = {c: i + 1 for i, c in enumerate(cols2)}
    
    order_mismatches = [
        c for c in common_cols if col1_idx[c] != col2_idx[c]
    ]
    
    # Build complete side-by-side DataFrame
    all_cols = cols1 + only_in_file2
    
    rows = []
    for c in all_cols:
        in_1 = c in set1
        in_2 = c in set2
        
        pos1 = col1_idx.get(c, None)
        pos2 = col2_idx.get(c, None)
        
        if in_1 and in_2:
            status = "In Both"
            order_status = "✓ Same" if pos1 == pos2 else f"≠ Diff ({pos1} vs {pos2})"
        elif in_1:
            status = "Only in File 1"
            order_status = "-"
        else:
            status = "Only in File 2"
            order_status = "-"
            
        rows.append({
            "Column Name": c,
            "Presence Status": status,
            "Pos in File 1": pos1 if pos1 is not None else "-",
            "Pos in File 2": pos2 if pos2 is not None else "-",
            "Order Match": order_status
        })
        
    if HAS_PANDAS:
        comparison_df = pd.DataFrame(rows)
        diff_df = comparison_df[comparison_df["Presence Status"] != "In Both"].copy().reset_index(drop=True)
    else:
        comparison_df = rows
        diff_df = [r for r in rows if r["Presence Status"] != "In Both"]
    
    result = {
        "file1_name": path1.name,
        "file2_name": path2.name,
        "file1_path": str(path1),
        "file2_path": str(path2),
        "is_exact_match": is_exact_match,
        "is_same_set": is_same_set,
        "total_cols1": len(cols1),
        "total_cols2": len(cols2),
        "common_count": len(common_cols),
        "only1_count": len(only_in_file1),
        "only2_count": len(only_in_file2),
        "order_mismatch_count": len(order_mismatches),
        "cols1": cols1,
        "cols2": cols2,
        "common_cols": common_cols,
        "only_in_file1": only_in_file1,
        "only_in_file2": only_in_file2,
        "order_mismatches": order_mismatches,
        "comparison_df": comparison_df,
        "diff_df": diff_df
    }
    
    if display_results:
        _display_comparison_report(result)
        
    return result


def _display_comparison_report(res: dict):
    """Renders formatted HTML / Markdown report for the comparison."""
    # Verdict banner
    if res["is_exact_match"]:
        verdict_html = f"""
        <div style="background-color: #d4edda; color: #155724; padding: 14px 18px; border-radius: 8px; border: 1px solid #c3e6cb; margin-bottom: 14px;">
            <h3 style="margin: 0 0 4px 0;">🟢 EXACT MATCH: Column Lists Are Identical!</h3>
            <p style="margin: 0; font-size: 14px;">Both files have the exact same <b>{res['total_cols1']}</b> columns in the exact same order.</p>
        </div>
        """
    elif res["is_same_set"]:
        verdict_html = f"""
        <div style="background-color: #fff3cd; color: #856404; padding: 14px 18px; border-radius: 8px; border: 1px solid #ffeeba; margin-bottom: 14px;">
            <h3 style="margin: 0 0 4px 0;">🟡 SAME COLUMNS, DIFFERENT ORDER</h3>
            <p style="margin: 0; font-size: 14px;">Both files contain the exact same set of <b>{res['total_cols1']}</b> columns, but their positions differ ({res['order_mismatch_count']} columns at different indices).</p>
        </div>
        """
    else:
        verdict_html = f"""
        <div style="background-color: #f8d7da; color: #721c24; padding: 14px 18px; border-radius: 8px; border: 1px solid #f5c6cb; margin-bottom: 14px;">
            <h3 style="margin: 0 0 4px 0;">🔴 COLUMN MISMATCH: Files Have Different Columns</h3>
            <p style="margin: 0; font-size: 14px;">Found <b>{res['only1_count']}</b> column(s) exclusive to File 1 and <b>{res['only2_count']}</b> column(s) exclusive to File 2. (Shared in common: <b>{res['common_count']}</b>).</p>
        </div>
        """
        
    display(HTML(verdict_html))
    
    # Summary Table
    summary_data = {
        "Metric": [
            "📁 File 1",
            "📁 File 2",
            "Total Columns in File 1",
            "Total Columns in File 2",
            "🔗 Shared Common Columns",
            "🔵 Columns ONLY in File 1",
            "🟠 Columns ONLY in File 2",
            "Exact Match (Same Names & Order)?",
            "Same Set of Columns (Ignoring Order)?"
        ],
        "Value": [
            f"<code>{res['file1_name']}</code>",
            f"<code>{res['file2_name']}</code>",
            res["total_cols1"],
            res["total_cols2"],
            res["common_count"],
            res["only1_count"],
            res["only2_count"],
            "✅ YES" if res["is_exact_match"] else "❌ NO",
            "✅ YES" if res["is_same_set"] else "❌ NO"
        ]
    }
    display(HTML("<h3>📊 Executive Summary</h3>"))
    if HAS_PANDAS:
        display(pd.DataFrame(summary_data).set_index("Metric"))
    else:
        for m, v in zip(summary_data["Metric"], summary_data["Value"]):
            print(f"  {m:<40}: {v}")
    
    # Detailed differences
    if res["only1_count"] > 0:
        display(HTML(f"<h3>🔵 Columns Exclusive to File 1 (<code>{res['file1_name']}</code>) [{res['only1_count']}]:</h3>"))
        if HAS_PANDAS:
            df_only1 = pd.DataFrame({
                "Pos in File 1": [res["cols1"].index(c) + 1 for c in res["only_in_file1"]],
                "Column Name": res["only_in_file1"]
            })
            display(df_only1)
        else:
            for c in res["only_in_file1"]:
                print(f"  - [{res['cols1'].index(c) + 1}] {c}")
        
    if res["only2_count"] > 0:
        display(HTML(f"<h3>🟠 Columns Exclusive to File 2 (<code>{res['file2_name']}</code>) [{res['only2_count']}]:</h3>"))
        if HAS_PANDAS:
            df_only2 = pd.DataFrame({
                "Pos in File 2": [res["cols2"].index(c) + 1 for c in res["only_in_file2"]],
                "Column Name": res["only_in_file2"]
            })
            display(df_only2)
        else:
            for c in res["only_in_file2"]:
                print(f"  - [{res['cols2'].index(c) + 1}] {c}")
        
    if res["is_same_set"] and not res["is_exact_match"]:
        display(HTML(f"<h3>⚠️ Common Columns with Shifted Index Order [{res['order_mismatch_count']}]:</h3>"))
        mismatch_rows = []
        for c in res["order_mismatches"][:30]:
            mismatch_rows.append({
                "Column Name": c,
                "Position in File 1": res["cols1"].index(c) + 1,
                "Position in File 2": res["cols2"].index(c) + 1
            })
        if HAS_PANDAS:
            df_order = pd.DataFrame(mismatch_rows)
            display(df_order)


## 📂 3. Browse Available Catalog Files in `data/`
Run the cell below to inspect the catalog files in your `data/` directory and copy their relative paths.


In [4]:
# Scan and list all tabular files in the data/ directory
available_files = list_data_files()


,#,Relative Path,Filename,Format,Size (MB)
0,1,data/LAMOST_catalog/asu.fit,asu.fit,FIT,36.52 MB
1,2,data/apogee_catalogs/allStarLite-dr17-synspec_...,allStarLite-dr17-synspec_rev1.fits,FITS,1749.96 MB
2,3,data/extra_catalogs/K2_APG_GALAH.csv,K2_APG_GALAH.csv,CSV,13.70 MB
3,4,data/extra_catalogs/K2_Kepler_TESS_x_GALAH_LAM...,K2_Kepler_TESS_x_GALAH_LAMOST.csv,CSV,33.19 MB
4,5,data/extra_catalogs/K2_TESS_GALAH.csv,K2_TESS_GALAH.csv,CSV,26.59 MB
5,6,data/extra_catalogs/K2_kepler_x_LAMOST.csv,K2_kepler_x_LAMOST.csv,CSV,5.23 MB
6,7,data/extra_catalogs/K2_x_GALAH.csv,K2_x_GALAH.csv,CSV,23.84 MB
7,8,data/extra_catalogs/K2_x_LAMOST.csv,K2_x_LAMOST.csv,CSV,3.36 MB
8,9,data/extra_catalogs/K2apg_x_LAMOST.csv,K2apg_x_LAMOST.csv,CSV,2.98 MB
9,10,data/extra_catalogs/K2galahbhm_x_LAMOST.csv,K2galahbhm_x_LAMOST.csv,CSV,1.50 MB


## 4. Compare columns in Two Files
Specify the two file paths you want to compare in the cell below and run it.

> **Tip:** You can enter paths relative to this notebook (e.g. `../data/first_match/...`), relative to repo root (e.g. `data/first_match/...`), or just copy paths from the table above.


In [5]:
# ==============================================================================
# CONFIGURATION: Choose any two files to compare
# ==============================================================================

FILE_1 = "../data/extra_catalogs/K2galahbhm_x_LAMOST.csv"
FILE_2 = "../data/willetxlamost/K2galahbhm_x_LAMOST.csv"


# Perform comparison and display report
results = compare_columns(FILE_1, FILE_2)


,Value
Metric,
📁 File 1,<code>K2galahbhm_x_LAMOST.csv</code>
📁 File 2,<code>K2galahbhm_x_LAMOST.csv</code>
Total Columns in File 1,94
Total Columns in File 2,94
🔗 Shared Common Columns,94
🔵 Columns ONLY in File 1,0
🟠 Columns ONLY in File 2,0
Exact Match (Same Names & Order)?,✅ YES
Same Set of Columns (Ignoring Order)?,✅ YES


## 🔍 5. Side-by-Side Detailed Alignment & Filter
Inspect the differences or search for specific columns across both files.


In [6]:
# Display the differences table (all columns present in only one of the files)
diff_df = results["diff_df"]

if hasattr(diff_df, 'empty') and not diff_df.empty:
    display(HTML(f"<h4>Differences Table ({len(diff_df)} columns differing):</h4>"))
    display(diff_df)
elif isinstance(diff_df, list) and diff_df:
    display(HTML(f"<h4>Differences Table ({len(diff_df)} columns differing):</h4>"))
    display(diff_df)
else:
    display(HTML("<h4>✨ No differing columns! Both files have identical column sets.</h4>"))


In [7]:
# Search for specific columns by name or pattern (e.g. 'Gaia', 'fe_h', 'teff', 'sobject')
search_term = "Gaia"

if HAS_PANDAS:
    matching = results["comparison_df"][
        results["comparison_df"]["Column Name"].str.contains(search_term, case=False, na=False)
    ].reset_index(drop=True)
    display(HTML(f"<h4>Columns matching '<code>{search_term}</code>' ({len(matching)} found):</h4>"))
    display(matching)
else:
    matching = [r for r in results["comparison_df"] if search_term.lower() in r["Column Name"].lower()]
    print(f"Found {len(matching)} columns matching '{search_term}'")


,Column Name,Presence Status,Pos in File 1,Pos in File 2,Order Match
0,GAIADR3_ID,In Both,4,4,✓ Same
1,Gaia,In Both,80,80,✓ Same


## 🔍 6. Memory-Safe Column Comparison (Lightweight Streaming)
Compare two columns within the same table/file row-by-row, accounting for empty/NaN cells.
- **Ultra-lightweight streaming**: Reads CSV rows line-by-line (O(1) RAM, < 5 MB) or uses FITS `memmap=True`, preventing high RAM usage or VS Code crashes on multi-gigabyte files.
- Distinguishes **matching values**, **both empty/NaN**, **differing values**, and **only one column empty**.


In [8]:
# ==============================================================================
# TOOL: Compare Two Columns in the Same Table (Memory-Safe Streaming)
# ==============================================================================

def compare_table_columns(source, col1: str, col2: str, sample_size: int = 10):
    """
    Ultra-lightweight column comparison.
    Streams CSV rows line-by-line (< 5 MB RAM) or uses FITS memmap to prevent out-of-memory crashes.
    Accounts for empty/NaN cells.
    """
    total = 0
    val_match, both_empty, val_diff, only1_empty, only2_empty = 0, 0, 0, 0, 0
    samples = []
    empty_set = {'', 'nan', 'none', 'null', 'nan', '<na>'}

    # 1. In-memory DataFrame mode
    if hasattr(source, 'columns'):
        s1, s2 = source[col1], source[col2]
        total = len(source)
        e1 = s1.isna() | s1.astype(str).str.strip().str.lower().isin(empty_set)
        e2 = s2.isna() | s2.astype(str).str.strip().str.lower().isin(empty_set)
        both_empty = int((e1 & e2).sum())
        only1_empty = int((e1 & ~e2).sum())
        only2_empty = int((~e1 & e2).sum())
        both_filled = ~e1 & ~e2
        try:
            n1, n2 = pd.to_numeric(s1[both_filled]), pd.to_numeric(s2[both_filled])
            val_match = int(((n1 == n2) | ((n1 - n2).abs() < 1e-7)).sum())
        except Exception:
            val_match = int((s1[both_filled] == s2[both_filled]).sum())
        val_diff = int(both_filled.sum()) - val_match

        # Slice only top sample rows to prevent RAM explosion
        diff_mask = (both_filled & (s1 != s2)) | (e1 & ~e2) | (~e1 & e2)
        diff_idx = source[diff_mask].index[:sample_size]
        for idx in diff_idx:
            v1, v2 = s1.loc[idx], s2.loc[idx]
            st = "Values Differ" if (both_filled.loc[idx] and v1 != v2) else (f"Only '{col2}' filled" if e1.loc[idx] else f"Only '{col1}' filled")
            samples.append({"Row": idx, col1: v1, col2: v2, "Status": st})

    else:
        # 2. File mode
        path = resolve_file_path(source)
        if str(path).lower().endswith(('.fits', '.fit')) and HAS_ASTROPY:
            from astropy.io import fits
            import numpy as np
            with fits.open(path, memmap=True) as hdul:
                for hdu in hdul:
                    if hasattr(hdu, 'columns') and hdu.columns is not None:
                        if col1 in hdu.columns.names and col2 in hdu.columns.names:
                            c1, c2 = hdu.data[col1], hdu.data[col2]
                            break
            total = len(c1)
            e1 = np.isnan(c1) if np.issubdtype(c1.dtype, np.floating) else np.zeros(total, dtype=bool)
            e2 = np.isnan(c2) if np.issubdtype(c2.dtype, np.floating) else np.zeros(total, dtype=bool)
            both_empty = int(np.sum(e1 & e2))
            only1_empty = int(np.sum(e1 & ~e2))
            only2_empty = int(np.sum(~e1 & e2))
            both_filled = ~e1 & ~e2
            val_match = int(np.sum(both_filled & (c1 == c2)))
            val_diff = int(np.sum(both_filled)) - val_match
            diff_idx = np.where((both_filled & (c1 != c2)) | (e1 & ~e2) | (~e1 & e2))[0][:sample_size]
            for idx in diff_idx:
                samples.append({"Row": int(idx), col1: c1[idx], col2: c2[idx], "Status": "Values Differ" if both_filled[idx] else "Empty mismatch"})
        else:
            # 3. CSV streaming line-by-line (Constant < 5 MB RAM even on 10+ GB CSVs)
            with open(path, 'r', encoding='utf-8', errors='replace') as f:
                reader = csv.reader(f)
                header = next(reader)
                if col1 not in header or col2 not in header:
                    raise KeyError(f"Columns not found: {[c for c in [col1, col2] if c not in header]}")
                i1, i2 = header.index(col1), header.index(col2)
                for row_idx, row in enumerate(reader):
                    total += 1
                    v1 = row[i1].strip() if i1 < len(row) else ''
                    v2 = row[i2].strip() if i2 < len(row) else ''
                    e1 = v1.lower() in empty_set
                    e2 = v2.lower() in empty_set

                    if e1 and e2:
                        both_empty += 1
                    elif e1:
                        only1_empty += 1
                        if len(samples) < sample_size:
                            samples.append({"Row": row_idx + 1, col1: "NaN", col2: v2, "Status": f"Only '{col2}' filled"})
                    elif e2:
                        only2_empty += 1
                        if len(samples) < sample_size:
                            samples.append({"Row": row_idx + 1, col1: v1, col2: "NaN", "Status": f"Only '{col1}' filled"})
                    elif v1 == v2:
                        val_match += 1
                    else:
                        try:
                            n1, n2 = float(v1), float(v2)
                            if n1 == n2 or abs(n1 - n2) < 1e-7:
                                val_match += 1
                            else:
                                val_diff += 1
                                if len(samples) < sample_size:
                                    samples.append({"Row": row_idx + 1, col1: v1, col2: v2, "Status": "Values Differ"})
                        except ValueError:
                            val_diff += 1
                            if len(samples) < sample_size:
                                samples.append({"Row": row_idx + 1, col1: v1, col2: v2, "Status": "Values Differ"})

    # Summary table
    pct = lambda count: f"{count / total * 100:.2f}%" if total > 0 else "0.00%"
    summary_df = pd.DataFrame({
        "Category": [
            "Total Rows",
            "✅ Matching Values (Both Filled)",
            "⚪ Both Empty / NaN",
            "❌ Differing Values (Both Filled)",
            "⚠️ Only Col 1 is Empty",
            "⚠️ Only Col 2 is Empty"
        ],
        "Count": [total, val_match, both_empty, val_diff, only1_empty, only2_empty],
        "Percentage": ["100.0%", pct(val_match), pct(both_empty), pct(val_diff), pct(only1_empty), pct(only2_empty)]
    })

    display(HTML(f"<h3>📊 Comparison: <code>{col1}</code> vs <code>{col2}</code></h3>"))
    display(summary_df.set_index("Category"))

    # Display discrepancies preview if any
    if samples:
        diff_df = pd.DataFrame(samples).set_index("Row")
        display(HTML(f"<h4>Discrepancies Preview (showing up to {sample_size} rows):</h4>"))
        display(diff_df)
    else:
        display(HTML("<h4>🟢 All rows match! No discrepancies found between the two columns.</h4>"))


# --- CONFIGURE AND RUN ---
# Can be a file path (even a 2GB+ CSV/FITS) or an in-memory DataFrame
TABLE_SOURCE = "../data/galah_catalogs/galah_and_galah3dnlte.csv"
COLUMN_1 = "flag_sp_1"
COLUMN_2 = "flag_sp_2"

compare_table_columns(TABLE_SOURCE, COLUMN_1, COLUMN_2)


KeyboardInterrupt: 